In [ ]:
import cv2
import PIL.Image

# Load MJPG Videos frame by frame, in either rgb or grayscale
class MJPGVideoLoader:

    def __init__(self, path, mode='gray'):
        self.path = path
        self.video_stream = cv2.VideoCapture(str(self.path))
        if not self.video_stream.isOpened():
            raise IOError(f"Cannot open video: {self.path}")

        self.mode = mode if mode in ('gray', 'rgb') else 'gray'

    @property
    def fps(self):
        return self.video_stream.get(cv2.CAP_PROP_FPS)

    @property
    def frame_size(self):
        return (int(self.video_stream.get(cv2.CAP_PROP_FRAME_WIDTH)),
                int(self.video_stream.get(cv2.CAP_PROP_FRAME_HEIGHT)))

    def next_frame(self):
        ret, frame = self.video_stream.read()
        if not ret:
            print("Stream ended, or otherwise failed to read frame.")
            return None

        if self.mode == 'gray':
            # Convert the image to greyscale
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        elif self.mode == 'rgb':
            # Convert the image to rgb
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Convert to a pillow image
        image = Image.fromarray(frame)
        return image

    def __del__(self):
        self.video_stream.release()

In [ ]:
import numpy as np
from PIL import Image, ImageDraw
from scipy import ndimage as ndi
from skimage import filters, morphology, measure
import os


# Constants for working with the tierpsy worm motel video files
REFERENCE_SIZE = 2048
WELL_BOXES = [
    (70, 70, 900, 880),      # top-left
    (1020, 95, 1830, 895),   # top-right
    (50, 1000, 895, 1820),   # bottom-left
    (1000, 1020, 1825, 1815) # bottom-right
]

# Convert an image with multiple petri dishes into a collection of images, each of which contains only one dish
def extract_dishes(img, boxes=WELL_BOXES, reference_size=REFERENCE_SIZE):
    # Find the pixel delta
    sx = img.width / reference_size
    sy = img.height / reference_size
    wells = []
    # For each petri dish...
    for x0, y0, x1, y1 in boxes:
        # Create a subimage containing just that petri dish, and add it to the result
        box = (
            max(0, round(x0 * sx)), max(0, round(y0 * sy)),
            min(img.width, round(x1 * sx)), min(img.height, round(y1 * sy)),
        )
        wells.append(img.crop(box))

    # Return all well images
    return wells

# Convert the image to an np array that's grayscale 
def _image_to_grayscale_np_arr(img):
    if img.mode.startswith("I") or img.mode == "F":
        arr = np.asarray(img, dtype=float)
        lo, hi = np.percentile(arr, (1, 99))
        return np.clip((arr - lo) / max(hi - lo, 1e-9) * 255, 0, 255)
    return np.asarray(img.convert("L"), dtype=float)

# Helper function (LLM HELP) that is used to determine which structures are worms
def _thin_dark_structures(gray, tophat_radius, n_std):
    smoothed = filters.gaussian(gray, sigma=1, preserve_range=True)
    darkness = morphology.black_tophat(smoothed, morphology.disk(tophat_radius))
    is_dark = darkness > darkness.mean() + n_std * darkness.std()
 
    without_hairs = morphology.opening(is_dark, morphology.disk(2))
    return morphology.closing(without_hairs, morphology.disk(3))
 
# Find the shortest distance to any border of the image, from any edge of the bounding box
def _distance_to_border(bbox, image_shape):
    top, left, bottom, right = bbox
    height, width = image_shape
    return min(top, left, height - bottom, width - right)

def _plate_walls(gray, min_wall_fraction=0.01):
    blurred = filters.gaussian(gray, sigma=6, preserve_range=True)
    is_dark = blurred < filters.threshold_otsu(blurred)
 
    labels = measure.label(is_dark)
    region_sizes = np.bincount(labels.ravel())
    is_wall_region = region_sizes >= min_wall_fraction * gray.size
    is_wall_region[0] = False
    return is_wall_region[labels]

def _is_clear_of_walls(box, distance_to_wall, wall_margin):
    x0, y0, x1, y1 = box
    return distance_to_wall[y0:y1, x0:x1].min() >= wall_margin

# Determine the number of lines which intersect. This is used to help determine if it's a worm because a worm has no intersections
def _count_junctions(skeleton):
    neighbor_counts = ndi.convolve(skeleton.astype(int), np.ones((3, 3), int), mode="constant") - 1
    return ((neighbor_counts >= 3) & skeleton).sum()
 
# Add Padding to a Bounding Box
def _padded_box(bbox, pad, image_shape):
    top, left, bottom, right = bbox
    height, width = image_shape
    return (max(0, left - pad), max(0, top - pad), min(width, right + pad), min(height, bottom + pad))

def _draw_boxes(img, boxes, box_color, box_width):
    if img.mode.startswith("I") or img.mode == "F":
        annotated = Image.fromarray(_image_to_grayscale_np_arr(img).astype(np.uint8)).convert("RGB")
    else:
        annotated = img.convert("RGB")
    draw = ImageDraw.Draw(annotated)
    for box in boxes:
        draw.rectangle(box, outline=box_color, width=box_width)
    return annotated


# Algorithmically (NO AI) find the bounding boxes of the worms
def find_worm_boxes(
    well_img,
    min_area=200,
    max_area=3000,
    min_length=40,        # skeleton length in px
    min_width=7.5,        # mean width in px (area / skeleton length); hairs are thinner
    max_width=14,
    max_junctions=2,      # skeleton branch pixels allowed; debris/tangles have more
    edge_margin=25,       # ignore objects this close to the well border (px)
    tophat_radius=12,     # should exceed half the worm's width
    n_std=3.0,            # lower = more sensitive, higher = fewer false hits
    pad=4,                # grow each box slightly so thin tails are included
):
    gray = _image_to_grayscale_np_arr(well_img)
    candidates = _thin_dark_structures(gray, tophat_radius, n_std)

    boxes = []
    for region in measure.regionprops(measure.label(candidates)):
        if not min_area <= region.area <= max_area:
            continue
        if _distance_to_border(region.bbox, gray.shape) < edge_margin:
            continue

        skeleton = morphology.skeletonize(region.image)
        length = skeleton.sum()
        mean_width = region.area / max(length, 1)
        if length < min_length or not min_width <= mean_width <= max_width:
            continue
        if _count_junctions(skeleton) > max_junctions:
            continue

        boxes.append(_padded_box(region.bbox, pad, gray.shape))
    return boxes

def find_worm_boxes_in_plate(plate_img, wall_margin=25, **kwargs):
    """
    Like `find_worm_boxes`, but for a whole plate image with no splitting.
    Detected objects closer than `wall_margin` px to a wall are discarded.
    Extra keyword arguments (n_std, min_width, ...) go to `find_worm_boxes`.
    """
    walls = _plate_walls(_image_to_grayscale_np_arr(plate_img))
    distance_to_wall = ndi.distance_transform_edt(~walls)
 
    boxes = find_worm_boxes(plate_img, **kwargs)
    return [box for box in boxes
            if _is_clear_of_walls(box, distance_to_wall, wall_margin)]
 
 
def annotate_plate(plate_img, box_color=(255, 0, 0), box_width=3, **kwargs):
    """
    Find worms anywhere in a whole plate image and draw a box around each.
    Extra keyword arguments (wall_margin, n_std, ...) go to
    `find_worm_boxes_in_plate`.
 
    Returns
    -------
    PIL.Image.Image
        RGB copy of the plate with boxes drawn around detected worms.
    """
    boxes = find_worm_boxes_in_plate(plate_img, **kwargs)
    return _draw_boxes(plate_img, boxes, box_color, box_width)



def annotate_worms(well_img, box_color=(255, 0, 0), box_width=3, **kwargs):
    alledged_worm_bounding_boxes = find_worm_boxes(well_img, **kwargs)

    if well_img.mode.startswith("I") or well_img.mode == "F":
        annotated = Image.fromarray(_image_to_grayscale_np_arr(well_img).astype(np.uint8)).convert("RGB")
    else:
        annotated = well_img.convert("RGB")

    # Create the output image
    draw = ImageDraw.Draw(annotated)
    for box in alledged_worm_bounding_boxes:
        # Add each bounding box
        draw.rectangle(box, outline=box_color, width=box_width)

    # Return the annotated image
    return annotated

def analyze_frame(frame, label):
    os.makedirs(label, exist_ok=True)
    frame.save(f"{label}/RAW.png")

    annotated_plate = annotate_plate(frame)
    annotated_plate.save(f"{label}/ANNOTATED.png")

    wells = extract_dishes(frame)
    for i, well in enumerate(wells):
        well.save(f"{label}/WELL_{i}.png")
        annotated = annotate_worms(well)
        annotated.save(f"{label}/WELL_{i}_ANNOTATED.png")

def analyze_demo_frames():
    video = 'data/WORM_MOTEL/RawVideos/WORM_MOTEL.mjpg'
    vid_parser = MJPGVideoLoader(video)
    for i in range(5):
        analyze_frame(vid_parser.next_frame(), f"TEST_FRAME_#{i}")

# analyze_demo_frames()

In [ ]:
import argparse
import csv

import cv2
import numpy as np
from PIL import Image, ImageDraw

RED = (255, 0, 0)

def write_boxes_csv(boxes_by_frame, csv_path, fps=None):
    """
    Write one row per box: frame, [time_s,] x0, y0, x1, y1.

    boxes_by_frame maps frame index -> list of (x0, y0, x1, y1).
    time_s is included only when fps is given. Frames without worms
    produce no rows.
    """
    columns = ["frame", "x0", "y0", "x1", "y1"]
    if fps:
        columns.insert(1, "time_s")

    with open(csv_path, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=columns)
        writer.writeheader()
        for frame_index, boxes in sorted(boxes_by_frame.items()):
            for x0, y0, x1, y1 in boxes:
                row = {"frame": frame_index, "x0": x0, "y0": y0, "x1": x1, "y1": y1}
                if fps:
                    row["time_s"] = round(frame_index / fps, 3)
                writer.writerow(row)


def _frames(loader):
    while (frame := loader.next_frame()) is not None:
        yield frame


def _draw_boxes_on_frame(frame, boxes, box_color, box_width):
    annotated = frame.copy()
    draw = ImageDraw.Draw(annotated)
    for box in boxes:
        draw.rectangle(box, outline=box_color, width=box_width)
    return annotated


def _to_bgr_array(rgb_image):
    return cv2.cvtColor(np.asarray(rgb_image), cv2.COLOR_RGB2BGR)


def annotate_video(
    input_path,
    output_path,
    csv_path=None,
    detect=find_worm_boxes_in_plate,
    frame_step=1,
    box_color=RED,
    box_width=3,
    fourcc="MJPG",
):
    loader = MJPGVideoLoader(input_path, mode="rgb")
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*fourcc),
                             loader.fps, loader.frame_size)
    if not writer.isOpened():
        raise IOError(f"Cannot write video with codec '{fourcc}': {output_path}")

    boxes_by_frame = {}
    latest_boxes = []
    try:
        for frame_index, frame in enumerate(_frames(loader)):
            if frame_index % frame_step == 0:
                latest_boxes = detect(frame)
                boxes_by_frame[frame_index] = latest_boxes
            annotated = _draw_boxes_on_frame(frame, latest_boxes, box_color, box_width)
            writer.write(_to_bgr_array(annotated))
    finally:
        writer.release()

    if csv_path:
        write_boxes_csv(boxes_by_frame, csv_path, loader.fps)
    return boxes_by_frame


In [ ]:
# Annotate a whole video, outputing the new, annotated video as well as a csv with bounding box info broken down by frame
def demo_video_annotation():
    os.makedirs('out', exist_ok=True)
    i_video = 'data/WORM_MOTEL/RawVideos/WORM_MOTEL.mjpg'
    o_video = 'out/output.mjpg'
    results = annotate_video(i_video, o_video, 'out/bounding_boxes.csv')
    box_count = sum(len(boxes) for boxes in results.values())
    print(f"{len(results)} frames analyzed, {box_count} boxes found")

# demo_video_annotation()

In [ ]:
from IPython.display import HTML, display
import base64

def display_synced_videos(path1, path2, width=640):
    def encode(path):
        with open(path, "rb") as f:
            return base64.b64encode(f.read()).decode()

    v1 = encode(path1)
    v2 = encode(path2)

    html = f"""
    <div>
        <video id="v1" width="{width}" controls muted>
            <source src="data:video/x-motion-jpeg;base64,{v1}">
        </video>

        <video id="v2" width="{width}" controls muted>
            <source src="data:video/x-motion-jpeg;base64,{v2}">
        </video>
    </div>

    <script>
    const v1 = document.getElementById("v1");
    const v2 = document.getElementById("v2");

    // Keep them synchronized
    v1.addEventListener("play", () => v2.play());
    v1.addEventListener("pause", () => v2.pause());

    v1.addEventListener("seeked", () => {{
        v2.currentTime = v1.currentTime;
    }});

    v2.addEventListener("play", () => v1.play());
    v2.addEventListener("pause", () => v1.pause());

    // Correct small timing differences
    setInterval(() => {{
        if (Math.abs(v1.currentTime - v2.currentTime) > 0.03) {{
            v2.currentTime = v1.currentTime;
        }}
    }}, 100);
    </script>
    """

    display(HTML(html))

display_synced_videos("data/WORM_MOTEL/RawVideos/WORM_MOTEL.mjpg", "out/output.mjpg", width = 500)